In [44]:
# =========================================================
# Faster runs distribution + weighted transition summarizer
# =========================================================
import math
from functools import lru_cache
from typing import Any, Optional, Tuple

import numpy as np
import pandas as pd

# runs distribution 계산을 위한 helper 함수들
# - _normalize_binary_state: 다양한 형태의 T/F 값을 bool 또는 None으로 정규화
# - _two_sided_p_from_z: z-score로부터 양측 p-value 계산
# - _runs_pmf_cached: n_t, n_f에 대한 runs PMF를 계산하고 캐싱
# - runs_exact_p_values: 관측된 runs 수에 대한 정확한 p-value 계산

# 다양한 형태의 T/F 값을 bool 또는 None으로 정규화
def _normalize_binary_state(x: Any) -> Optional[bool]:
    if pd.isna(x):
        return None

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    if isinstance(x, (int, np.integer, float, np.floating)):
        if x == 1:
            return True
        if x == 0:
            return False

    s = str(x).strip().upper()
    true_set = {"1", "T", "TRUE", "Y", "YES"}
    false_set = {"0", "F", "FALSE", "N", "NO"}

    if s in true_set:
        return True
    if s in false_set:
        return False

    raise ValueError(f"Could not normalize binary state: {x!r}")

# z-score로부터 양측 p-value 계산
def _two_sided_p_from_z(z: float) -> float:
    if pd.isna(z):
        return np.nan
    return math.erfc(abs(z) / math.sqrt(2))


@lru_cache(maxsize=None) # n_t, n_f에 대한 runs PMF를 캐싱하여 반복 계산을 빠르게 함, 메모리 사용량은 입력 범위에 따라 달라질 수 있음
def _runs_pmf_cached(n_t: int, n_f: int) -> Tuple[np.ndarray, np.ndarray]:
    n_t = int(n_t)
    n_f = int(n_f)

    if n_t < 0 or n_f < 0:
        raise ValueError("n_t and n_f must be nonnegative.")

    if n_t + n_f == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    if n_t == 0 or n_f == 0:
        return np.array([0], dtype=int), np.array([1.0], dtype=float)

    total = math.comb(n_t + n_f, n_t)
    ks = []
    probs = []

    r_max = 2 * min(n_t, n_f) + (1 if n_t != n_f else 0)

    for r in range(2, r_max + 1):
        if r % 2 == 0:
            m = r // 2
            ways = 2 * math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m - 1)
        else:
            m = (r - 1) // 2
            ways = 0
            if m <= n_t - 1 and m - 1 <= n_f - 1:
                ways += math.comb(n_t - 1, m) * math.comb(n_f - 1, m - 1)
            if m - 1 <= n_t - 1 and m <= n_f - 1:
                ways += math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m)

        if ways > 0:
            ks.append(r - 1)
            probs.append(ways / total)

    ks_arr = np.asarray(ks, dtype=int)
    probs_arr = np.asarray(probs, dtype=float)

    if probs_arr.size:
        probs_arr = probs_arr / probs_arr.sum()

    return ks_arr, probs_arr

# 관측된 runs 수에 대한 정확한 p-value 계산
def runs_exact_p_values(k_obs: int, n_t: int, n_f: int):
    ks, probs = _runs_pmf_cached(int(n_t), int(n_f))
    if ks.size == 0:
        return (np.nan, np.nan, np.nan)

    e_k = 2 * n_t * n_f / (n_t + n_f)

    p_lower = probs[ks <= k_obs].sum()
    p_upper = probs[ks >= k_obs].sum()

    obs_dist = abs(k_obs - e_k)
    p_two = probs[np.abs(ks - e_k) >= obs_dist - 1e-12].sum()

    return float(p_lower), float(p_upper), float(min(1.0, p_two))

# Main function1: analyze_runs_from_weighted_df - 전체/범주/문서용
# weighted sentence-level df에서 run 관련 통계를 계산하는 함수
def analyze_runs_from_weighted_df(
    df: pd.DataFrame,
    *,
    unit_col: str = "docu_id",
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    count_col: str = "count",
    add_exact_p: bool = True,
) -> pd.DataFrame:
    """
    weighted sentence-level df에서 run 관련 통계를 계산한다.

    전제
    ----
    - 각 row는 '현재 문장 상태' 1개를 대표한다.
    - count_col은 그런 row가 몇 개 있었는지를 뜻한다.
    - state_col은 현재 문장 T/F
    - prev_state_col은 이전 문장 T/F
    - has_prev_col은 실제 이전 문장 존재 여부
    """

    work = df.copy()

    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)

    # has_prev가 False면 prev값은 계산에서 쓰면 안 됨
    work.loc[work["_has_prev"] != True, "_prev"] = None

    # --------------------------
    # 1. 전체 상태 수: 모든 문장 기준
    # --------------------------
    curr_counts = (
        work.groupby([unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    curr_pivot = curr_counts.pivot_table(
        index=unit_col,
        columns="_curr",
        values=count_col,
        fill_value=0,
    )
    # 
    all_units = pd.Index(sorted(work[unit_col].dropna().unique()), name=unit_col)
    out = pd.DataFrame(index=all_units)

    out["n_T"] = curr_pivot[True] if True in curr_pivot.columns else 0.0
    out["n_F"] = curr_pivot[False] if False in curr_pivot.columns else 0.0
    out["n"] = out["n_T"] + out["n_F"]

    # --------------------------
    # 2. sequence 개수
    #    has_prev == False 인 현재문장은 각 sequence의 시작
    # --------------------------
    seq_counts = (
        work.loc[work["_has_prev"] == False]
        .groupby(unit_col, observed=True)[count_col]
        .sum()
    )

    out["n_sequences"] = seq_counts.reindex(out.index, fill_value=0).astype(float)

    # --------------------------
    # 3. 전이표: has_prev == True 인 pair만
    # --------------------------
    pair_work = work.loc[
        (work["_has_prev"] == True)
        & (work["_prev"].notna())
        & (work["_curr"].notna())
    ].copy()

    pair_counts = (
        pair_work.groupby([unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    pair_pivot = pair_counts.pivot_table(
        index=unit_col,
        columns=["_prev", "_curr"],
        values=count_col,
        fill_value=0,
    )

    def get_pair(prev_val, curr_val):
        if len(pair_pivot) == 0:
            return pd.Series(0.0, index=out.index)
        if (prev_val, curr_val) in pair_pivot.columns:
            return pair_pivot[(prev_val, curr_val)].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    out["a_TT"] = get_pair(True, True)
    out["b_TF"] = get_pair(True, False)
    out["c_FT"] = get_pair(False, True)
    out["d_FF"] = get_pair(False, False)

    out["k"] = out["b_TF"] + out["c_FT"]
    out["n_pairs"] = out["a_TT"] + out["b_TF"] + out["c_FT"] + out["d_FF"]

    # 이론적으로는 n - n_sequences 와 같아야 함
    out["expected_n_pairs_from_sequences"] = out["n"] - out["n_sequences"]
    out["pair_count_matches_structure"] = (
        out["n_pairs"] == out["expected_n_pairs_from_sequences"]
    )

    # --------------------------
    # 4. 전이율
    # --------------------------
    out["transition_rate"] = out["k"] / out["n_pairs"].replace(0, np.nan) # 전체 pair 대비 전환이 일어난 pair의 비율
    out["TF_transition_rate"] = out["b_TF"] / (out["a_TT"] + out["b_TF"]).replace(0, np.nan)
    out["FT_transition_rate"] = out["c_FT"] / (out["c_FT"] + out["d_FF"]).replace(0, np.nan)

    # --------------------------
    # 5. runs 기대값 / 분산 / z
    #    기본 runs 공식은 한 sequence 기준
    # --------------------------
    n = out["n"].replace(0, np.nan)

    out["E_k"] = 2 * out["n_T"] * out["n_F"] / n # runs의 기대값
    out["expected_transition_rate"] = out["E_k"]/ out["n_pairs"].replace(0, np.nan) # 기대 전환율
    out["전환의확률비"] = out["transition_rate"]/ out["expected_transition_rate"].replace(0, np.nan) # 실제 전환율이 기대 전환율의 몇 배인지

    out["Var_k"] = (
        2 * out["n_T"] * out["n_F"] * (2 * out["n_T"] * out["n_F"] - n)
    ) / (n**2 * (n - 1))

    out.loc[out["n"] <= 1, "Var_k"] = np.nan
    out["z_k"] = (out["k"] - out["E_k"]) / np.sqrt(out["Var_k"]) # runs 수의 z-score
    out["p_two_sided_z"] = out["z_k"].map(_two_sided_p_from_z) # z-score로부터 양측 p-value 계산

    # --------------------------
    # 6. exact p-value
    #    한 unit이 한 sequence일 때만 엄밀함
    # --------------------------
    if add_exact_p:
        p_lower = []
        p_upper = []
        p_two = []

        for _, row in out.iterrows():
            if row["n_sequences"] != 1:
                p_lower.append(np.nan)
                p_upper.append(np.nan)
                p_two.append(np.nan)
                continue

            vals = runs_exact_p_values(
                int(row["k"]),
                int(row["n_T"]),
                int(row["n_F"]),
            )
            p_lower.append(vals[0])
            p_upper.append(vals[1])
            p_two.append(vals[2])

        out["p_lower_exact"] = p_lower
        out["p_upper_exact"] = p_upper
        out["p_two_exact"] = p_two

    # --------------------------
    # 7. 2x2 association
    # --------------------------
    N = out["n_pairs"].replace(0, np.nan)
    out["E_a"] = (out["a_TT"] + out["b_TF"]) * (out["a_TT"] + out["c_FT"]) / N
    out["pearson_a"] = (out["a_TT"] - out["E_a"]) / np.sqrt(out["E_a"])

    a = out["a_TT"] + 0.5
    b = out["b_TF"] + 0.5
    c = out["c_FT"] + 0.5
    d = out["d_FF"] + 0.5

    out["OR"] = (a * d) / (b * c)
    out["log_OR"] = np.log(out["OR"])

    return out.reset_index()

# Main function2: analyze_transitions_against_baseline : 동사/어미/보조용언용
# baseline_col 기준 기대 전환율과 unit_col별 실제 전환율을 비교하는 함수
def analyze_transitions_against_baseline(
    df: pd.DataFrame,
    *,
    baseline_col: str,
    unit_col: str,
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    count_col: str = "count",
) -> pd.DataFrame:
    """
    동사/어미/보조용언 등 특정 항목별 전이율을,
    baseline_col 기준 기대 전환율과 비교한다.

    예
    ----
    baseline_col = "문서범주"
    unit_col = "V_form"

    의미
    ----
    - 기대 전환율은 문서범주별 전체 구조에서 계산
    - 실제 전환율은 문서범주 × V_form별로 계산
    - 전환의확률비 = 실제 전환율 / baseline 기대 전환율
    """

    work = df.copy()

    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)

    # has_prev가 False이면 prev는 전이 계산에서 제외
    work.loc[work["_has_prev"] != True, "_prev"] = None

    # -------------------------------------------------
    # 1. baseline 계산
    #    기대 전환율은 baseline_col 기준으로 고정
    # -------------------------------------------------
    baseline = analyze_runs_from_weighted_df(
        work,
        unit_col=baseline_col,
        state_col=state_col,
        prev_state_col=prev_state_col,
        has_prev_col=has_prev_col,
        count_col=count_col,
        add_exact_p=False,
    )

    baseline_cols = [
        baseline_col,
        "n_T",
        "n_F",
        "n",
        "n_sequences",
        "n_pairs",
        "transition_rate",
        "expected_transition_rate",
        "E_k",
        "Var_k",
        "z_k",
        "OR",
        "log_OR",
    ]

    baseline = baseline[baseline_cols].rename(
        columns={
            "n_T": "baseline_n_T",
            "n_F": "baseline_n_F",
            "n": "baseline_n",
            "n_sequences": "baseline_n_sequences",
            "n_pairs": "baseline_n_pairs",
            "transition_rate": "baseline_transition_rate",
            "expected_transition_rate": "baseline_expected_transition_rate",
            "E_k": "baseline_E_k",
            "Var_k": "baseline_Var_k",
            "z_k": "baseline_z_k",
            "OR": "baseline_OR",
            "log_OR": "baseline_log_OR",
        }
    )

    # -------------------------------------------------
    # 2. unit별 실제 전이표 계산
    #    여기서는 E_k, Var_k를 새로 만들지 않음
    # -------------------------------------------------
    pair_work = work.loc[
        (work["_has_prev"] == True)
        & (work["_prev"].notna())
        & (work["_curr"].notna())
        & (work[unit_col].notna())
        & (work[baseline_col].notna())
    ].copy()

    g = (
        pair_work
        .groupby([baseline_col, unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    pivot = g.pivot_table(
        index=[baseline_col, unit_col],
        columns=["_prev", "_curr"],
        values=count_col,
        fill_value=0,
    )

    out = pd.DataFrame(index=pivot.index)

    def get_pair(prev_val, curr_val):
        if (prev_val, curr_val) in pivot.columns:
            return pivot[(prev_val, curr_val)].astype(float)
        return pd.Series(0.0, index=pivot.index)

    out["a_TT"] = get_pair(True, True)
    out["b_TF"] = get_pair(True, False)
    out["c_FT"] = get_pair(False, True)
    out["d_FF"] = get_pair(False, False)

    out["k"] = out["b_TF"] + out["c_FT"]
    out["n_pairs"] = out["a_TT"] + out["b_TF"] + out["c_FT"] + out["d_FF"]

    out["transition_rate"] = out["k"] / out["n_pairs"].replace(0, np.nan)
    out["TF_transition_rate"] = out["b_TF"] / (out["a_TT"] + out["b_TF"]).replace(0, np.nan)
    out["FT_transition_rate"] = out["c_FT"] / (out["c_FT"] + out["d_FF"]).replace(0, np.nan)

    # unit 자체의 T/F 분포도 참고용으로 계산
    curr_counts = (
        work.loc[
            work[unit_col].notna()
            & work[baseline_col].notna()
            & work["_curr"].notna()
        ]
        .groupby([baseline_col, unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    curr_pivot = curr_counts.pivot_table(
        index=[baseline_col, unit_col],
        columns="_curr",
        values=count_col,
        fill_value=0,
    )

    out["unit_n_T"] = curr_pivot[True].reindex(out.index, fill_value=0).astype(float) if True in curr_pivot.columns else 0.0
    out["unit_n_F"] = curr_pivot[False].reindex(out.index, fill_value=0).astype(float) if False in curr_pivot.columns else 0.0
    out["unit_n"] = out["unit_n_T"] + out["unit_n_F"]
    out["unit_T_ratio"] = out["unit_n_T"] / out["unit_n"].replace(0, np.nan)

    out = out.reset_index()

    # -------------------------------------------------
    # 3. baseline 기대값 붙이기
    # -------------------------------------------------
    out = out.merge(
        baseline,
        on=baseline_col,
        how="left",
    )

    # -------------------------------------------------
    # 4. baseline 기준 비교 지표
    # -------------------------------------------------
    out["expected_transition_rate"] = out["baseline_expected_transition_rate"]

    out["transition_OE_ratio"] = (
        out["transition_rate"] / out["expected_transition_rate"].replace(0, np.nan)
    )

    out["transition_suppression_rate"] = 1 - out["transition_OE_ratio"]

    out["persistence_rate"] = 1 - out["transition_rate"]
    out["expected_persistence_rate"] = 1 - out["expected_transition_rate"]

    out["persistence_OE_ratio"] = (
        out["persistence_rate"] / out["expected_persistence_rate"].replace(0, np.nan)
    )

    # -------------------------------------------------
    # 5. unit 내부 2x2 연관성
    #    이건 기대전환률과 별개로 참고 가능
    # -------------------------------------------------
    a = out["a_TT"] + 0.5
    b = out["b_TF"] + 0.5
    c = out["c_FT"] + 0.5
    d = out["d_FF"] + 0.5

    out["unit_OR"] = (a * d) / (b * c)
    out["unit_log_OR"] = np.log(out["unit_OR"])

    return out

In [19]:
# =========================================================
# Imports for logodds and chi2 functions
# =========================================================
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))

from stats.pivots import make_context_outcome_tab, make_unit_context_binary_pivot #이 함수들은 피벗 테이블을 만들어주는 함수로, logodds나 chi2 계산 전에 데이터를 준비하는 단계에서 사용됨.
from stats.logodds import logodds_context_by_outcome_from_tab, logodds_unit_by_context_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 logodds 계산을 수행하는 함수로, 실제로 logodds 계산을 하는 단계에서 사용됨.
from stats.chi2_utils import chi2_overall_from_tab, chi2_by_unit_from_pivot #이 함수들은 피벗 테이블을 입력으로 받아서 카이제곱 검정을 수행하는 함수로, 실제로 chi2 계산을 하는 단계에서 사용됨.


In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np

CSV_PATH = r"..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260410_15-29.csv"
        #FILTER_BODY_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False) & (df['speaker'] == 'body')]

COUNT_MODE = "weight" #가중치 파일
WEIGHT_COL="count" #입력 파일이 이미 가중치가 적용된 형태이므로, 가중치 컬럼을 지정하여 "개수" 대신 "가중치 합계"로 계산하도록 함.

df = pd.read_csv(CSV_PATH)

print(f"CSV 파일 로드 완료: {len(df):,}, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
df.columns

CSV 파일 로드 완료: 1,022,399, now: 2026.05.03_23:56:12


Index(['Unnamed: 0', 'category', '매체', 'file_id', 'docu_id', 'speaker',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
       'p

In [20]:
df['dominant_EN_No'].value_counts()
#df.loc[df['dominant_EN_No'].isna(), 'category'].value_counts()

dominant_EN_No
 1101.0    939004
 1171.0     52181
 1401.0     14726
-1.0        12448
 1411.0      1305
 1271.0      1146
 3002.0       443
 1221.0       354
 1451.0       156
 1331.0       126
 1301.0       101
 1999.0        73
 1179.0        59
 1241.0        54
 1231.0        35
 1202.0        34
 3001.0        29
 1431.0        23
 1372.0        22
 701.0         18
 1167.0        18
 1373.0        11
 1409.0         6
 1105.0         5
 10.0           5
 70.0           3
 1421.0         3
 1432.0         2
 999.0          2
 20.0           2
 110.0          2
 90.0           2
 151.0          1
Name: count, dtype: int64

In [6]:
#document 파일 읽기
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)

print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_docu: {df_docu.columns.tolist()}")


파일 읽기 완료 - 2026.05.03_23:24:14
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count', 'docu_dominant_ratio', 'docu_sent_count', 'docu_head_count', 'docu_body_count', 'docu_body_has_E_count', 'docu_body_not_quote_count', 'docu_body_not_quote_and_었_count', 'docu_었_결합_오즈비', 'docu_었_결합_로그오즈비', 'docu_었_결합_등급', 'docu_었_결합_성향', 'category2', 'file_sent_count', 'file_head_count', 'file_body_count', 'file_body_has_E_count', 'file_body_not_quote_count', 'file_body_not_quote_and_었_count', 'file_었_결합_오즈비', 'file_었_결합_로그오즈비', 'file_었_결합_등급', 'file_었_결합_성향']


In [13]:
df_docu['category'].value_counts()


category
보도해설    14729
인문사회    10265
사설       1631
자연       1517
체험기술     1469
칼럼       1357
허구일반     1229
총류        727
허구아동      231
Name: count, dtype: int64

In [8]:
df.drop(columns="category", inplace=True)  

In [21]:
df["prev_sentence_f_EP_M"].value_counts(dropna=False)

prev_sentence_f_EP_M
False    1011644
True       10755
Name: count, dtype: int64

In [15]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

cols = [c for c in df_docu.columns if c != "file_id"]
df = safe_merge(df, df_docu, "docu_id", cols)

print(f"merge 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(df.columns.to_list())

merge 완료 - 2026.05.04_07:56:57
['Unnamed: 0', 'category', '매체', 'file_id', 'docu_id', 'speaker', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No', 'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No', 'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label', 'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form', 'prev_sentence_f_EP_form', 'next_sentence_f_EP_form', 'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote', 'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT', 'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T', 'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T', 'prev_sentence_f_EP_M', 'next_sentence_f_EP_TT', 'next_sentence_f_EP_T', 'n

In [16]:
df['문서범주'] = df['category'].map({
    '보도해설':'신문', 
    '사설': '신문', 
    '칼럼': '신문', 
    '인문사회': '책', 
    '체험기술': '체험', 
    '허구일반': '허구', 
    '자연': '책', 
    '총류': '책', 
    '허구아동': '허구'
})
df['문서범주'].value_counts()


문서범주
책     369276
허구    317007
신문    247299
체험     88817
Name: count, dtype: int64

In [33]:
df.loc[df["has_prev_sentence"] == False,]["prev_sentence_f_EP_T"].value_counts(dropna=False)

prev_sentence_f_EP_T
False    41961
Name: count, dtype: int64

In [32]:
from stats.filtering import apply_filters, FilterValue, has_value, _topn_values
df['total'] = True
df_1101 = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF"})
df_V = apply_filters(df, {"VX0_No": -1})
print(f"{len(df_1101):,} rows with f_EN_No=1101, label='EF'(다EF)),\n {len(df_V):,} rows with VX0_No=-1 (V),\n {len(df):,} total rows")

834,511 rows with f_EN_No=1101, label='EF'(다EF)),
 716,183 rows with VX0_No=-1 (V),
 1,022,399 total rows


In [59]:
df["prev_sentence_f_EP_T"].value_counts(dropna=False), df["has_prev_sentence"].value_counts(dropna=False)

(prev_sentence_f_EP_T
 False    616858
 True     421698
 Name: count, dtype: int64,
 has_prev_sentence
 True     980438
 False     41961
 NaN       16157
 Name: count, dtype: int64)

In [42]:
df["prev_sentence_f_EP_T"].isnull().count()

np.int64(1038556)

In [31]:
df.head(5)

,Unnamed: 0,category,매체,file_id,docu_id,speaker,sen_count,sen_count_has_E,sen_count_not_quote,sen_count_has_E_not_quote,...,file_head_count,file_body_count,file_body_has_E_count,file_body_not_quote_count,file_body_not_quote_and_었_count,file_었_결합_오즈비,file_었_결합_로그오즈비,file_었_결합_등급,file_었_결합_성향,문서범주
0,0,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,8.0,...,164,1226,1083,951,260,0.520896,-0.652204,2,회피,신문
1,1,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,8.0,...,164,1226,1083,951,260,0.520896,-0.652204,2,회피,신문
2,2,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,8.0,...,164,1226,1083,951,260,0.520896,-0.652204,2,회피,신문
3,3,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,8.0,...,164,1226,1083,951,260,0.520896,-0.652204,2,회피,신문
4,4,보도해설,신문,AA0001,AA0001.001,body,10,9.0,9.0,8.0,...,164,1226,1083,951,260,0.520896,-0.652204,2,회피,신문


In [45]:
MOAD = "RUNS"
UNIT_COL = "file_id"#'total'
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")



out_df = analyze_runs_from_weighted_df(
    df_V,
    unit_col = UNIT_COL,
    state_col = "sentence_f_EP_T",
    prev_state_col = "prev_sentence_f_EP_T",
    has_prev_col = "has_prev_sentence",
    count_col = "count",
    add_exact_p = True,
)
print(f"analyze_runs_from_weighted_df 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

# =========================================================
# 4. save to file
# =========================================================
                                    
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"

file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
out_df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")


C:\Users\yu2hy\AppData\Local\Temp\ipykernel_33036\1778334743.py:140: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  work.loc[work["_has_prev"] != True, "_prev"] = None


analyze_runs_from_weighted_df 완료 - 2026.05.05_09:40:54
***저장합니다.:    2026-05-05 09:40:54.096395***
Output file for pivot table: ..\csv\결과값\tense\RUNS_by_file_id_2026-05-05_09-40.csv
